# 📄 Varalaxmi Jangili — Document Analyst
## Multimodal Crime / Incident Report Analyzer

**Data Type:** Police reports / official incident documents (PDF)  
**Objective:** Extract structured info from PDFs → CSV with report ID, type, date, location, summary

### Pipeline:
1. Download/load PDF documents
2. Extract text with pdfplumber / PyMuPDF
3. OCR for scanned documents (pytesseract)
4. Named Entity Recognition with spaCy
5. Output structured CSV


In [1]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q pdfplumber pymupdf pytesseract spacy pandas requests
!python -m spacy download en_core_web_sm
!apt-get install -y -q tesseract-ocr > /dev/null 2>&1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# =============================================
# CELL 2: Import Libraries
# =============================================
import pdfplumber
import fitz  # PyMuPDF
import pytesseract
import spacy
import pandas as pd
import requests
import os
import re
import warnings
warnings.filterwarnings('ignore')

nlp = spacy.load("en_core_web_sm")
os.makedirs("pdf_data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("All libraries loaded!")


All libraries loaded!


In [3]:
# =============================================
# CELL 3: Load PDF
# =============================================
import os, shutil

if os.path.exists("pdf_data"):
    shutil.rmtree("pdf_data")
os.makedirs("pdf_data")

shutil.copy("/content/LESO2.pdf", "pdf_data/LESO2.pdf")

print(f"Loaded: {[f for f in os.listdir('pdf_data') if not f.startswith('.')]}")

Loaded: ['LESO2.pdf']


In [4]:
# =============================================
# CELL 4: PDF Text Extraction Functions
# =============================================

def extract_text_pdfplumber(pdf_path):
    """Extract text using pdfplumber (best for structured/table PDFs)."""
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"  pdfplumber error: {e}")
    return text.strip()

def extract_text_pymupdf(pdf_path):
    """Extract text using PyMuPDF (fast, good fallback)."""
    text = ""
    try:
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text() + "\n"
        doc.close()
    except Exception as e:
        print(f"  PyMuPDF error: {e}")
    return text.strip()

def extract_text_ocr(pdf_path):
    """Extract text from scanned PDFs using OCR."""
    text = ""
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc[page_num]
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for better OCR
            img_path = f"/tmp/page_{page_num}.png"
            pix.save(img_path)
            from PIL import Image
            img = Image.open(img_path)
            page_text = pytesseract.image_to_string(img)
            text += page_text + "\n"
        doc.close()
    except Exception as e:
        print(f"  OCR error: {e}")
    return text.strip()

def smart_extract(pdf_path):
    """Try pdfplumber first, fall back to PyMuPDF, then OCR."""
    text = extract_text_pdfplumber(pdf_path)
    method = "pdfplumber"

    if len(text.strip()) < 50:
        text = extract_text_pymupdf(pdf_path)
        method = "PyMuPDF"

    if len(text.strip()) < 50:
        text = extract_text_ocr(pdf_path)
        method = "OCR"

    return text, method

# Test extraction
print("Testing PDF extraction methods...")
test_files = sorted([f for f in os.listdir("pdf_data") if f.endswith(".pdf")])
for f in test_files[:1]:
    text, method = smart_extract(os.path.join("pdf_data", f))
    print(f"\n{f} (method: {method}):")
    print(text[:300] + "...")


Testing PDF extraction methods...

LESO2.pdf (method: pdfplumber):
To: Whom it may Concern
From: Fort Smith Police Department
Date: January 19, 2015
Ref: MRAP
The following documentation is intended to document the intended use and training of the Mine Resistatant
Ambush Protection Vehicle, which was allocated to the Fort Smith Police Department, by the United Stat...


In [5]:
# =============================================
# CELL 5: Information Extraction with spaCy + Regex
# =============================================

def classify_doc_severity(text):
    """Classify severity based on keywords in document text."""
    text_lower = text.lower()

    high_keywords = ["weapon", "tactical", "armored", "mrap", "rifle",
                     "explosive", "swat", "fatal", "shooting", "homicide"]
    medium_keywords = ["training", "equipment", "vehicle", "arrest",
                       "enforcement", "patrol", "incident", "investigation"]

    for kw in high_keywords:
        if kw in text_lower:
            return "High"
    for kw in medium_keywords:
        if kw in text_lower:
            return "Medium"
    return "Low"

def extract_report_info(text, filename, page_num=0):
    """Extract structured information from police report text."""
    doc = nlp(text[:100000])  # Limit for spaCy

    # Extract entities
    locations = list(set(ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC", "FAC"]))
    persons = list(set(ent.text for ent in doc.ents if ent.label_ == "PERSON"))
    orgs = list(set(ent.text for ent in doc.ents if ent.label_ == "ORG"))
    dates = list(set(ent.text for ent in doc.ents if ent.label_ == "DATE"))

    # Extract department name
    dept_match = re.search(r'([\w\s]+(?:Police Department|Sheriff[\'\u2019]?s?\s*(?:Office|Department)))', text, re.IGNORECASE)
    department = dept_match.group(1).strip() if dept_match else (orgs[0] if orgs else "Unknown")

    # Extract document type
    doc_types = ["Training Plan", "1033", "MRAP", "Policy", "Lesson Plan",
                 "Incident Report", "SOP", "Memorandum", "Aircraft Request", "Invoice"]
    incident_type = "General Report"
    text_lower = text.lower()
    for dt in doc_types:
        if dt.lower() in text_lower:
            incident_type = dt
            break

    # Extract date
    date_match = re.search(r'(?:Date|DATE)[:\s]*(\w+\s+\d{1,2},?\s*\d{4}|\d{1,2}/\d{1,2}/\d{2,4})', text)
    if not date_match:
        date_match = re.search(r'(\w+\s+\d{1,2},?\s*\d{4})', text)
    report_date = date_match.group(1).strip() if date_match else (dates[0] if dates else "Unknown")

    # Extract officer/contact
    officer_match = re.search(r'(?:Submitted by|Prepared by|Respectfully|Sincerely)[,:\s]*\n*\s*([A-Z][\w\s\.]+)', text)
    officer = officer_match.group(1).strip()[:50] if officer_match else "Unknown"

    # Location
    location = ", ".join(locations[:3]) if locations else "Arkansas"

    # Summary (first 200 chars)
    summary = text[:200].replace('\n', ' ').strip()

    # Report ID
    report_id = f"RPT_{page_num+1:03d}"

    return {
        "Report_ID": report_id,
        "Department": department[:60],
        "Incident_Type": incident_type,
        "Date": report_date,
        "Location": location,
        "Officer": officer,
        "Severity": classify_doc_severity(text),
        "Summary": summary,
        "Entities_Persons": ", ".join(persons[:5]),
        "Entities_Orgs": ", ".join(orgs[:5])
    }

# Process all PDFs — split by pages for multi-page PDFs
print("Extracting information from all PDFs...")
pdf_files = sorted([f for f in os.listdir("pdf_data") if f.endswith(".pdf")])
results = []

for filename in pdf_files:
    filepath = os.path.join("pdf_data", filename)
    print(f"\nProcessing: {filename}")

    # Extract text per page
    try:
        doc_pdf = fitz.open(filepath)
        print(f"  Pages: {len(doc_pdf)}")

        # Group pages into sections (each department = ~1-3 pages)
        current_text = ""
        section_count = 0

        for page_num in range(len(doc_pdf)):
            page_text = doc_pdf[page_num].get_text()

            # Check if this is a new department section
            is_new_section = bool(re.search(
                r'(?:Police Department|Sheriff[\'\u2019]?s?\s*(?:Office|Department)|MEMORANDUM|To Whom)',
                page_text, re.IGNORECASE
            )) and len(current_text) > 200

            if is_new_section and current_text:
                info = extract_report_info(current_text, filename, section_count)
                results.append(info)
                section_count += 1
                print(f"  -> Section {section_count}: {info['Department'][:40]} | {info['Incident_Type']}")
                current_text = page_text
            else:
                current_text += "\n" + page_text

        # Last section
        if current_text.strip():
            info = extract_report_info(current_text, filename, section_count)
            results.append(info)
            section_count += 1
            print(f"  -> Section {section_count}: {info['Department'][:40]} | {info['Incident_Type']}")

        doc_pdf.close()
        print(f"  Total sections extracted: {section_count}")

    except Exception as e:
        print(f"  Error: {e}")
        text, method = smart_extract(filepath)
        info = extract_report_info(text, filename, 0)
        results.append(info)

# Reassign Report IDs
for i, r in enumerate(results):
    r["Report_ID"] = f"RPT_{i+1:03d}"

df_documents = pd.DataFrame(results)
print(f"\nProcessed {len(df_documents)} document sections.")
df_documents

Extracting information from all PDFs...

Processing: LESO2.pdf
  Pages: 75
  -> Section 1: Fort Smith Police Department | MRAP
  -> Section 2: Fort Smith Police Department | MRAP
  -> Section 3: 1
Hot Springs Police Department | MRAP
  -> Section 4: 2
Hot Springs Police Department | MRAP
  -> Section 5: 3
 
 
 
 
 
 
 
 
 
Hot Springs Police D | MRAP
  -> Section 6: which was allocated to the Jacksonville  | MRAP
  -> Section 7: The Jefferson County Sheriff’s Office | 1033
  -> Section 8: Lonoke County Sheriff’s Office | MRAP
  -> Section 9: Lonoke County Sheriff’s Office | MRAP
  -> Section 10: Lonoke County Sheriff’s Office | MRAP
  Total sections extracted: 10

Processed 10 document sections.


,Report_ID,Department,Incident_Type,Date,Location,Officer,Severity,Summary,Entities_Persons,Entities_Orgs
0,RPT_001,Fort Smith Police Department,MRAP,"January 19, 2015",Arkansas,Cpl. Monty McMillen \nFort Smith Police Depart...,High,To: Whom it may Concern From: Fort Smith Poli...,"McMillen, LESO Liaison","the Fort Smith Police Department, Fort Smith P..."
1,RPT_002,Fort Smith Police Department,MRAP,"June 30th, 2014",Arkansas,Unknown,High,COURSE: LESSON TITLE: CLEET ...,"Training, Driver, McMillen","FCMTC Maintenance, Fort Smith Police Departmen..."
2,RPT_003,1\nHot Springs Police Department,MRAP,Unknown,Arkansas,Ofc. Eric Wacaster \n \nRevisions,High,1 Hot Springs Police Department S.W.A.T. Train...,Eric Wacaster,"Hot Springs Police Department, MRAP Operations..."
3,RPT_004,2\nHot Springs Police Department,MRAP,Unknown,Rostan,Unknown,High,2 Hot Springs Police Department S.W.A.T. Train...,"Eric Wacaster, Zac Rostan","Commission, Force on Force, Instructor, Genera..."
4,RPT_005,3\n \n \n \n \n \n \n \n \n \nHot Springs Poli...,MRAP,Unknown,Arkansas,Unknown,High,3 Hot Springs Police Departm...,"Shut Down, Zac Rostan","Vehicle Recovery \n \n \n \nIII,..."
5,RPT_006,which was allocated to the Jacksonville AR Pol...,MRAP,"January 14, 2015",the United States \nGovernment,Lt. Brett Hibbs \nJacksonville Police Departme...,High,To: Whom it may Concern From: Lt. Brett Hibbs...,"Smith, Brett Hibbs \nDate, Brett Hibbs \n8.0 H...","Instructor, Operations of the Mine Resistant A..."
6,RPT_007,The Jefferson County Sheriff’s Office,1033,"January 13, 2015",• Mount,Unknown,High,Mine Resistant Ambush Protected (MRAP) Vehicle...,"MAINTENANCE, • Restricted, Vehicle, • Falls","• Steering, Vehicle \nStandard Operating Proce..."
7,RPT_008,Lonoke County Sheriff’s Office,MRAP,AR 72086,Lonoke County Sheriff’s Office,Captain David Bufford \nLonoke County Sheriff,High,Lonoke County Sheriff’s Office John Staley - ...,"LESO Liaison, David Bufford, John Staley - She...","The Lonoke County Sheriff’s Office, the United..."
8,RPT_009,Lonoke County Sheriff’s Office,MRAP,AR 72086,Lonoke County Sheriff’s Office,Unknown,High,Lonoke County Sheriff’s Office John Staley - ...,James Hall ...,CLEST Continuing Education Course \n \n \n M...
9,RPT_010,Lonoke County Sheriff’s Office,MRAP,AR 72086,"Lonoke County Sheriff’s Office, Sheriff",Unknown,High,Lonoke County Sheriff’s Office John Staley - ...,John Staley - Sheriff,Tactics / Operations \nSearch and Rescue \nEva...


In [6]:
# =============================================
# CELL 6: Generate Final Structured CSV
# =============================================

df_doc_output = df_documents.copy()

df_doc_output.rename(columns={
    "Incident_Type": "Doc_Type",
    "Summary": "Key_Detail"
}, inplace=True)

# Add Program column
def assign_program(row):
    text = (str(row.get("Key_Detail", "")) + " " + str(row.get("Department", ""))).lower()
    if any(kw in text for kw in ["mrap", "tactical", "weapon", "armored", "swat"]):
        return "Law Enforcement Support"
    elif any(kw in text for kw in ["training", "lesson", "course"]):
        return "Training Program"
    elif any(kw in text for kw in ["vehicle", "aircraft"]):
        return "Equipment Transfer"
    return "General"

df_doc_output["Program"] = df_doc_output.apply(assign_program, axis=1)

# Keep only required columns in correct order
df_doc_output = df_doc_output[["Report_ID", "Department", "Doc_Type", "Date", "Program", "Key_Detail"]]

df_doc_output.to_csv("outputs/document_analyst_output.csv", index=False)

print("=" * 70)
print("DOCUMENT ANALYST - FINAL OUTPUT")
print("=" * 70)
print(f"Total documents processed: {len(df_doc_output)}")
print(f"Output saved to: outputs/document_analyst_output.csv")
print("=" * 70)
df_doc_output


DOCUMENT ANALYST - FINAL OUTPUT
Total documents processed: 10
Output saved to: outputs/document_analyst_output.csv


,Report_ID,Department,Doc_Type,Date,Program,Key_Detail
0,RPT_001,Fort Smith Police Department,MRAP,"January 19, 2015",Law Enforcement Support,To: Whom it may Concern From: Fort Smith Poli...
1,RPT_002,Fort Smith Police Department,MRAP,"June 30th, 2014",Training Program,COURSE: LESSON TITLE: CLEET ...
2,RPT_003,1\nHot Springs Police Department,MRAP,Unknown,Law Enforcement Support,1 Hot Springs Police Department S.W.A.T. Train...
3,RPT_004,2\nHot Springs Police Department,MRAP,Unknown,Law Enforcement Support,2 Hot Springs Police Department S.W.A.T. Train...
4,RPT_005,3\n \n \n \n \n \n \n \n \n \nHot Springs Poli...,MRAP,Unknown,Training Program,3 Hot Springs Police Departm...
5,RPT_006,which was allocated to the Jacksonville AR Pol...,MRAP,"January 14, 2015",Law Enforcement Support,To: Whom it may Concern From: Lt. Brett Hibbs...
6,RPT_007,The Jefferson County Sheriff’s Office,1033,"January 13, 2015",Law Enforcement Support,Mine Resistant Ambush Protected (MRAP) Vehicle...
7,RPT_008,Lonoke County Sheriff’s Office,MRAP,AR 72086,General,Lonoke County Sheriff’s Office John Staley - ...
8,RPT_009,Lonoke County Sheriff’s Office,MRAP,AR 72086,Training Program,Lonoke County Sheriff’s Office John Staley - ...
9,RPT_010,Lonoke County Sheriff’s Office,MRAP,AR 72086,General,Lonoke County Sheriff’s Office John Staley - ...


### ✅ Document Analyst Complete!
**Output:** `outputs/document_analyst_output.csv`

| Column | Description |
|--------|-------------|
| Report_ID | Unique report identifier |
| Department | Police department name |
| Doc_Type | Type of document (MRAP, 1033, Training Plan, etc.) |
| Date | Report date |
| Program | Associated program (Law Enforcement Support, Training, etc.) |
| Key_Detail | Summary of key findings |



# THE END